# Functional-Law Compression v0.11
Synthetic model-selection study. Candidate temporal functional F is restricted to shared systemhood features B,C,A,P and selected derivatives. Goal: remove unnecessary freedom rather than improve fit at any cost.

In [ ]:
import numpy as np, itertools, matplotlib.pyplot as plt
rng=np.random.default_rng(19); t=np.linspace(0,10,601); n=len(t)
def sig(x): return 1/(1+np.exp(-x))
def features(on,off,k,phase=0):
    W=sig(k*(t-on))*sig(k*(off-t))
    A=.85-.55*W
    P=.78-.12*W+.025*np.sin(.7*t+phase)
    B=.55+.18*W
    C=.25+.48*W
    dA=np.gradient(A,t); dP=np.gradient(P,t)
    return dict(B=B,C=C,A=A,P=P,dA=dA,dP=dP,W=W)
specs=[(2.4,7.6,2.5,0),(3.2,6.8,9,.4),(2.0,7.1,5,.9),(3.0,7.0,7,1.4),(2.8,6.5,4,2.)]
data=[features(*s) for s in specs]
# hidden synthetic law is deliberately sparse: A + dA only
alpha_true={'A':.55,'dA':.32}; eps_true=2e-4; sigma=5e-5
for m,d in enumerate(data):
    F=alpha_true['A']*(d['A']-.5)+alpha_true['dA']*d['dA']
    nuisance=(.8+.25*m)*1e-4*d['W']+((-1)**m)*5e-5*np.exp(-.5*((t-(4.5+.15*m))/2.2)**2)+(m-2)*1e-5
    d['y']=eps_true*F+nuisance+rng.normal(0,sigma,n)
    d['D']=np.exp(-.5*((t-(4.5+.15*m))/2.2)**2)


## Candidate shared feature sets
The fitted temporal coefficients theta_j = epsilon alpha_j are global across experiments. Interaction/drift/offset nuisance coefficients remain experiment-specific.

In [ ]:
models=[('A',['A']),('A+dA',['A','dA']),('A+P+dA',['A','P','dA']),('BCAP',['B','C','A','P']),('BCAP+dA+dP',['B','C','A','P','dA','dP'])]
def build(feats,train_ids):
    M=len(train_ids); X=np.zeros((M*n,len(feats)+3*M)); y=np.zeros(M*n)
    for q,m in enumerate(train_ids):
        d=data[m]; sl=slice(q*n,(q+1)*n)
        for j,f in enumerate(feats): X[sl,j]=d[f]-(.5 if f in ['A','P','B','C'] else 0)
        X[sl,len(feats)+3*q]=d['W']; X[sl,len(feats)+3*q+1]=d['D']; X[sl,len(feats)+3*q+2]=1; y[sl]=d['y']
    return X,y
results=[]
for name,feats in models:
    X,y=build(feats,list(range(len(data)))); b=np.linalg.lstsq(X,y,rcond=None)[0]; r=y-X@b; rss=r@r; N=len(y); k=X.shape[1]; bic=N*np.log(rss/N)+k*np.log(N); results.append((name,bic,rss,np.linalg.cond(X),b[:len(feats)]))
for r in sorted(results,key=lambda z:z[1]): print(r)


## Leave-one-experiment-out predictive score

In [ ]:
for name,feats in models:
    rms=[]
    for hold in range(len(data)):
        train=[m for m in range(len(data)) if m!=hold]; X,y=build(feats,train); b=np.linalg.lstsq(X,y,rcond=None)[0]; theta=b[:len(feats)]
        d=data[hold]; T=np.column_stack([d[f]-(.5 if f in ['A','P','B','C'] else 0) for f in feats]); Nuis=np.column_stack([d['W'],d['D'],np.ones(n)]); nb=np.linalg.lstsq(Nuis,d['y']-T@theta,rcond=None)[0]; pred=T@theta+Nuis@nb; rms.append(np.sqrt(np.mean((d['y']-pred)**2)))
    print(name,'LOEO RMSE=',np.mean(rms))


## Feature collinearity
Highly correlated systemhood features cannot be assigned separate physical coefficients reliably.

In [ ]:
names=['B','C','A','P','dA','dP']; Z=np.column_stack([np.concatenate([d[f] for d in data]) for f in names]); R=np.corrcoef(Z,rowvar=False); print(names); print(np.round(R,2))
plt.figure(figsize=(6,5)); plt.imshow(R,vmin=-1,vmax=1); plt.xticks(range(len(names)),names); plt.yticks(range(len(names)),names); plt.colorbar(label='correlation'); plt.tight_layout(); plt.show()


## Sparse recovery sanity check
The synthetic generator used only A and dA. Model selection should prefer a compact model if the data contain enough information; failure to do so would expose non-identifiability.

In [ ]:
best=min(results,key=lambda z:z[1]); print('BIC-best model:',best[0]); print('true active basis: A + dA')


## Interpretation
A compact law is meaningful only if its coefficients are identifiable and stable across experiments. B,C,A can become strongly correlated because they are all generated from the same merger profile; in that regime coefficient interpretation is unreliable even when prediction is good. v0.11 therefore treats model compression and experimental feature diversity as inseparable.